# Credit Card Fraud Detection – Random Forest
**Member:** *IT22256928 - Nawarathne D G N M V K B*  
**Algorithm:** Random Forest Classifier  
**Dataset:** [Kaggle – Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

---

## Algorithm Background

### What is a Decision Tree?
A Decision Tree is a supervised learning algorithm that recursively partitions the feature space by asking yes/no questions about feature values. At each internal node, the algorithm selects the feature and threshold that best separates the two classes (measured by Gini impurity or information gain). The result is a tree of rules ending in leaf nodes that give a class prediction.

**Limitation:** A single deep decision tree tends to *overfit* — it memorises the training data perfectly but generalises poorly to unseen data. This is known as high variance.

### What is Random Forest?
Random Forest is an ensemble learning method that corrects the overfitting problem of individual trees using two key techniques:

1. **Bootstrap Aggregation (Bagging):** Each tree in the forest is trained on a different random sample of the training data, drawn *with replacement*. On average, each tree sees approximately 63% of the training rows; the remaining ~37% are called Out-Of-Bag (OOB) rows.

2. **Random Feature Subsets:** At each node split, only a random subset of features is considered (typically √p features, where p is the total number of features). This decorrelates the trees — each tree learns slightly different patterns, so their errors do not compound.

The final prediction is made by **majority vote** (classification) across all trees. Because each tree is different, individual errors cancel out — this is the core intuition behind ensemble methods.

### Why Random Forest for Credit Card Fraud Detection?
- **Handles class imbalance:** The `class_weight='balanced'` parameter automatically upweights fraud samples, compensating for the 0.17% fraud rate.
- **No feature scaling required:** Decision tree splits are based on thresholds, not distances — so the PCA-transformed V1–V28 features are handled natively.
- **Robust to noise:** Averaging hundreds of trees smooths out noise in the PCA features, which contain overlapping information about transaction patterns.
- **Feature importance:** The forest provides a reliable measure of which features are most predictive of fraud, enabling interpretability.
- **Strong performance:** Random Forest consistently outperforms single decision trees and logistic regression on tabular, high-dimensional datasets like this one.

**Reference:** Breiman, L. (2001). *Random Forests.* Machine Learning, 45(1), 5–32.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
# Standard data manipulation and visualisation libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-learn: Random Forest model and hyperparameter search
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Evaluation metrics — we avoid plain accuracy due to class imbalance
from sklearn.metrics import (
    classification_report,        # precision, recall, F1 per class
    confusion_matrix,              # TP, TN, FP, FN counts
    roc_auc_score,                 # area under the ROC curve
    roc_curve,                     # TPR/FPR at every threshold
    precision_recall_curve,        # precision/recall at every threshold
    average_precision_score,       # area under precision-recall curve
    f1_score                       # harmonic mean of precision and recall
)

sns.set_theme(style='whitegrid')
print('All libraries loaded successfully.')

All libraries loaded successfully.


## 1. Load Preprocessed Data

The shared preprocessing notebook (`00_preprocessing.ipynb`) has already:
- Scaled `Amount` and `Time` using StandardScaler
- Split the data 80/20 train/test with stratification
- Applied SMOTE to the **training set only** to balance the fraud/legit ratio
- Saved everything to `preprocessed_data.pkl`

We load that file here. The test set deliberately retains the real-world 0.17% fraud rate.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

X_train, X_test, y_train, y_test = joblib.load('/content/drive/MyDrive/preprocessed_data.pkl')
print(f'Training set  : {X_train.shape[0]:,} rows × {X_train.shape[1]} features')
print(f'Test set      : {X_test.shape[0]:,}  rows × {X_test.shape[1]} features')
print(f'\nTraining class balance (after SMOTE):')
print(pd.Series(y_train).value_counts().rename({0: 'Legitimate', 1: 'Fraud'}))
print(f'\nTest class balance (original imbalance):')
print(pd.Series(y_test).value_counts().rename({0: 'Legitimate', 1: 'Fraud'}))

Mounted at /content/drive
Training set  : 454,902 rows × 30 features
Test set      : 56,962  rows × 30 features

Training class balance (after SMOTE):
Class
Legitimate    227451
Fraud         227451
Name: count, dtype: int64

Test class balance (original imbalance):
Class
Legitimate    56864
Fraud            98
Name: count, dtype: int64


## 2. Hyperparameter Tuning with RandomizedSearchCV

A Random Forest has several key hyperparameters that significantly affect performance:

| Hyperparameter | What it controls | Effect |
|---|---|---|
| `n_estimators` | Number of trees | More trees = more stable, but slower |
| `max_depth` | Maximum depth of each tree | Deeper = more expressive, more overfitting risk |
| `min_samples_split` | Minimum rows required to split a node | Higher = simpler trees, less overfitting |
| `max_features` | Features considered at each split | `sqrt` means √30 ≈ 5-6 features per split |

**Why RandomizedSearchCV instead of GridSearchCV?**  
GridSearchCV tries every combination (3×4×3×2 = 72 combinations × 5 folds = 360 fits). RandomizedSearchCV randomly samples `n_iter=15` combinations × 5 folds = 75 fits. This is ~5× faster while still finding near-optimal parameters.  

We use `scoring='f1'` because optimising for accuracy on imbalanced data is misleading.

In [ ]:
# Define the hyperparameter search space
param_dist = {
    'n_estimators':      [100, 200, 300],          # number of trees in the forest
    'max_depth':         [None, 10, 20, 30],        # None = unlimited depth
    'min_samples_split': [2, 5, 10],                # minimum samples needed to split a node
    'max_features':      ['sqrt', 'log2']           # features considered per split
}

# Base model with class weighting to handle fraud imbalance
# class_weight='balanced' makes the model penalise fraud misclassification more heavily
rf_base = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',  # compensates for 0.17% fraud rate
    n_jobs=-1                 # use all CPU cores for parallel training
)

# Randomised search: tries 15 random combinations with 5-fold cross-validation
# Scoring = F1 because accuracy is misleading on imbalanced datasets
rand_search = RandomizedSearchCV(
    rf_base,
    param_distributions=param_dist,
    n_iter=15,           # number of random parameter combinations to try
    cv=5,                # 5-fold cross-validation
    scoring='f1',        # optimise for F1-score on the fraud class
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print('Starting RandomizedSearchCV — this may take a few minutes...')
rand_search.fit(X_train, y_train)

print(f'\nBest hyperparameters found: {rand_search.best_params_}')
print(f'Best cross-validation F1 score: {rand_search.best_score_:.4f}')

# Extract the best model for evaluation
rf = rand_search.best_estimator_

Starting RandomizedSearchCV — this may take a few minutes...
Fitting 5 folds for each of 15 candidates, totalling 75 fits


## 3. Predictions & Evaluation

We evaluate the model on the **original imbalanced test set** to simulate real-world conditions.  

Key metrics for imbalanced fraud detection:
- **ROC-AUC:** Probability that the model ranks a fraud transaction above a legitimate one. Threshold-independent.
- **Average Precision (AUPRC):** Area under the precision-recall curve. More informative than ROC-AUC for highly imbalanced datasets.
- **F1-Score:** Harmonic mean of precision and recall for the fraud class. Balances false positives (customer annoyance) and false negatives (financial loss).

In [ ]:
# Generate class predictions (0 or 1) and probability scores for the fraud class
y_pred = rf.predict(X_test)           # hard predictions using default 0.5 threshold
y_prob = rf.predict_proba(X_test)[:, 1]  # fraud probability for each transaction

# Full classification report: per-class precision, recall, F1, support
print('=' * 55)
print('CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

# Summary metrics
roc_auc = roc_auc_score(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)

print(f'ROC-AUC Score         : {roc_auc:.4f}  (1.0 = perfect, 0.5 = random)')
print(f'Average Precision     : {avg_prec:.4f}  (AUPRC — key metric for imbalanced data)')
print(f'F1-Score (Fraud)      : {f1:.4f}  (balance between catching fraud and false alarms)')

## 4. Visualisations

Three standard visualisations for binary fraud classification:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── (a) Confusion Matrix ──────────────────────────────────────────────────────
# Shows TP (correctly caught fraud), TN (correctly identified legit),
# FP (legitimate flagged as fraud = false alarm),
# FN (fraud missed = most costly error in practice)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title('Confusion Matrix – Random Forest', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

# ── (b) ROC Curve ──────────────────────────────────────────────────────────────
# Plots True Positive Rate vs False Positive Rate at every classification threshold.
# A perfect classifier has AUC = 1.0; random guessing gives AUC = 0.5 (dashed line).
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'Random Forest (AUC = {auc_score:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].set_title('ROC Curve – Random Forest', fontsize=12, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].legend(loc='lower right')

# ── (c) Precision-Recall Curve ────────────────────────────────────────────────
# More informative than ROC for heavily imbalanced datasets.
# High AP means the model achieves high precision across many recall levels.
prec, rec, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
axes[2].plot(rec, prec, color='chocolate', lw=2, label=f'Random Forest (AP = {ap:.4f})')
axes[2].axhline(y=y_test.mean(), color='gray', linestyle='--', lw=1,
                label=f'No-skill baseline ({y_test.mean():.4f})')
axes[2].set_title('Precision-Recall Curve – Random Forest', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend()

plt.tight_layout()
plt.savefig('rf_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as rf_results.png')

## 5. Feature Importances

Random Forest measures feature importance as the **mean decrease in Gini impurity** across all trees and all splits where the feature was used. Features with higher importance contribute more to separating fraud from legitimate transactions.

Since V1–V28 are PCA-transformed, the original feature names are anonymised — but we can still identify *which* transformed components are most discriminative.

In [ ]:
# Extract feature importances from the trained forest
fi_df = pd.DataFrame({
    'Feature':    X_test.columns,
    'Importance': rf.feature_importances_  # mean decrease in Gini impurity
})

# Sort by importance descending and keep top 15
fi_df = fi_df.sort_values('Importance', ascending=False).head(15)

# Plot horizontal bar chart
plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df, x='Importance', y='Feature', palette='Oranges_r')
plt.title('Top 15 Feature Importances – Random Forest\n(Mean Decrease in Gini Impurity)',
          fontsize=12, fontweight='bold')
plt.xlabel('Mean Decrease in Gini Impurity')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 most important features for fraud detection:')
print(fi_df.head(5).to_string(index=False))

## 6. Out-of-Bag (OOB) Score

The OOB score is a built-in validation mechanism of Random Forest. When each tree is trained on a bootstrap sample (~63% of the data), the remaining ~37% of rows are never seen by that tree. The model predicts those rows after training and averages the results across all trees to produce the OOB score.

This gives us a free cross-validation estimate **without** needing a separate validation set.

In [ ]:
# Train a separate model with OOB scoring enabled
# We use the best hyperparameters found above, but fix n_estimators to 200
# to keep runtime reasonable while still getting a reliable OOB estimate

# Safely extract best params, excluding n_estimators (we override that below)
best_params = {k: v for k, v in rand_search.best_params_.items()
               if k != 'n_estimators'}

rf_oob = RandomForestClassifier(
    n_estimators=200,       # enough trees for a stable OOB estimate
    oob_score=True,         # enable out-of-bag scoring
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
    **best_params           # apply remaining best hyperparameters
)

print('Training OOB model...')
rf_oob.fit(X_train, y_train)

print(f'\nOOB Score: {rf_oob.oob_score_:.4f}')
print('(OOB score is an unbiased estimate of generalisation accuracy)')
print('A high OOB score (>0.90) confirms the model generalises beyond the training data.')

## 7. Results Summary

Fill in the values below after running this notebook:

| Metric | Value |
|---|---|
| Best CV F1 (training) | *(copy from Section 2 output)* |
| ROC-AUC (test set) | *(copy from Section 3 output)* |
| Average Precision (test set) | *(copy from Section 3 output)* |
| F1-Score Fraud (test set) | *(copy from Section 3 output)* |
| OOB Score | *(copy from Section 6 output)* |
| Best hyperparameters | *(copy from Section 2 output)* |

**Interpretation:**
- A ROC-AUC close to 1.0 confirms the model reliably distinguishes fraud from legitimate transactions.
- Average Precision above 0.80 shows the model maintains high precision even as recall increases.
- The F1-score represents the operational balance between catching fraud (recall) and minimising false alarms (precision).

## 8. Critical Analysis and Discussion

### Strengths of Random Forest on This Dataset
- The ensemble approach significantly reduces the overfitting observed in a single Decision Tree.
- `class_weight='balanced'` effectively handles the extreme 0.17% fraud rate without distorting the test distribution.
- Feature importances provide interpretable insight into which PCA components are most predictive.
- RandomizedSearchCV found strong hyperparameters efficiently, exploring the search space in a fraction of the time GridSearchCV would require.

### Limitations
- **Interpretability:** While a single decision tree is fully human-readable, a forest of 200+ trees cannot be inspected directly. This is a significant limitation in regulated industries (e.g., banking) where models must be explainable to regulators.
- **Training time:** Random Forest is considerably slower to train than Logistic Regression, especially after SMOTE expands the training set.
- **Static model:** The model is trained once on historical data. Fraud patterns evolve over time (concept drift), so the model will degrade in production without regular retraining.

### How Could Accuracy Be Improved?

1. **Threshold tuning:** The default classification threshold of 0.5 may not be optimal for fraud detection. By moving the threshold lower (e.g., 0.3), recall improves at the cost of precision. The optimal threshold can be found from the precision-recall curve.

2. **Gradient Boosting (XGBoost / LightGBM):** Unlike Random Forest (which trains trees in parallel on independent samples), boosting builds trees sequentially where each tree corrects the errors of the previous one. This typically achieves higher precision-recall scores on fraud datasets.

3. **More data:** With only 492 fraud cases across 284,807 transactions, the model has limited examples to learn from. Acquiring more labeled fraud cases or using data augmentation techniques beyond SMOTE (e.g., ADASYN, BorderlineSMOTE) could improve detection of rare fraud patterns.

4. **Feature engineering:** The raw PCA features contain limited temporal or behavioural context. Adding features like rolling transaction frequency per card, time since last transaction, or merchant category could significantly improve fraud signal.

5. **SHAP values for explainability:** SHAP (SHapley Additive exPlanations) can decompose the model's predictions per transaction, identifying exactly which features pushed a specific transaction towards 'Fraud'. This addresses the interpretability limitation and would be essential in a production banking system.

6. **Cost-sensitive learning:** In practice, the cost of missing a fraud ($500 loss) is very different from the cost of a false alarm (customer inconvenience). A cost matrix can be incorporated into the model to explicitly optimise for business costs rather than statistical F1.

### Possible Future Work
- **Online learning:** Implement an incremental learning system that updates the model as new fraud patterns emerge.
- **Anomaly detection:** Combine supervised Random Forest with unsupervised anomaly detection (e.g., Isolation Forest) for a hybrid approach that can detect novel, never-before-seen fraud patterns.
- **Neural networks:** Deep learning models (e.g., autoencoders for anomaly detection, or feed-forward networks with focal loss) have shown strong results on this dataset in the literature.

## Appendix: Environment Information

In [ ]:
# Print library versions for reproducibility
import sklearn
print(f'Python:      (run python --version in terminal)')
print(f'NumPy:       {np.__version__}')
print(f'Pandas:      {pd.__version__}')
print(f'Scikit-learn:{sklearn.__version__}')
print(f'Seaborn:     {sns.__version__}')
print(f'Matplotlib:  {plt.matplotlib.__version__}')